# Find latest LLM cache entries

Show the most recent entries in the `llm_cache` pickle store, inspired by the cache/pickle analysis workflow.

In [11]:
from pathlib import Path
import pickle
from datetime import datetime

import pandas as pd

cache_dir = Path("/mnt/labstore/bespoke_olap/cache/llm_cache")

print(f"Using cache dir: {cache_dir}")

cache_files = sorted(
    cache_dir.glob("*.pkl"), key=lambda p: p.stat().st_mtime, reverse=True
)
print(f"Found {len(cache_files)} cache pickle files.")

Using cache dir: /mnt/labstore/bespoke_olap/cache/llm_cache
Found 2594 cache pickle files.


In [12]:
N_LAST = 10

rows = []
for p in cache_files[:N_LAST]:
    stat = p.stat()
    row = {
        "file": p.name,
        "modified": datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
        "size_kb": round(stat.st_size / 1024, 2),
        "ok": False,
        "cache_type": None,
        "response_id": None,
        "response_model": None,
        "total_tokens": None,
        "llm_time_s": None,
        "error": None,
    }

    try:
        with open(p, "rb") as f:
            obj = pickle.load(f)

        response = getattr(obj, "response", None)
        usage = getattr(response, "usage", None) if response is not None else None

        row["ok"] = True
        row["cache_type"] = type(obj).__name__
        row["response_id"] = (
            getattr(response, "id", None) if response is not None else None
        )
        row["response_model"] = (
            getattr(response, "model", None) if response is not None else None
        )
        row["total_tokens"] = (
            getattr(usage, "total_tokens", None) if usage is not None else None
        )
        row["llm_time_s"] = getattr(obj, "llm_time", None)
    except Exception as exc:
        row["error"] = str(exc)

    rows.append(row)

last10_df = pd.DataFrame(rows)
display(last10_df)

for f in last10_df["file"]:
    print(f)

# Optional: inspect the very latest object in detail
if cache_files:
    with open(cache_files[0], "rb") as f:
        latest_obj = pickle.load(f)
    latest_obj

,file,modified,size_kb,ok,cache_type,response_id,response_model,total_tokens,llm_time_s,error
0,13b347d1c7235b579b577e164f608993693b24dbeabdfe...,2026-03-05T20:38:01,45.91,True,LLMCacheType,resp_0c76abda3f84b97f0069a9e9a70dbc8199b943cbd...,gpt-5.3-codex,1487,2.980951,None
1,977ad10dcd77a1b5899e05b816e26c10213abd83f838ad...,2026-03-05T07:49:22,11.30,True,LLMCacheType,resp_06b93e7882a5e8be0069a9357feafc819b95edb65...,gpt-5.2-codex,1142,2.605387,None
2,2675206040f28983647b3612cb91e9cc9fa7e7a9715dd3...,2026-03-05T07:34:21,9.17,True,LLMCacheType,resp_0a1e5acb7f49f05c0069a931fb3b208199a36a2de...,gpt-5.2-codex,717,2.756559,None
3,ebf4fbe7fca036c17b6abc9b56d1299cfc55b98f0d8cda...,2026-03-05T00:35:36,329.49,True,LLMCacheType,resp_074d76b3cc1d78e50069a8cf37c2f4819b87fec12...,gpt-5.2-codex,124489,160.956209,None
4,9be1ca4e24afddcdb03035bdfd06ad887d796abfbf80b1...,2026-03-05T00:32:55,240.33,True,LLMCacheType,resp_074d76b3cc1d78e50069a8cf30ff94819b88bf92e...,gpt-5.2-codex,102022,6.513448,None
5,bfac5f109b8460fe371d90012c17088c35c8727356fa3a...,2026-03-05T00:32:48,230.19,True,LLMCacheType,resp_074d76b3cc1d78e50069a8cf2c4bd8819baad88c6...,gpt-5.2-codex,99406,4.472432,None
6,be578ede4f8c75524668755057616f86f11364331417ad...,2026-03-05T00:32:43,179.13,True,LLMCacheType,resp_074d76b3cc1d78e50069a8cf271b04819b8a3ecdf...,gpt-5.2-codex,87079,4.925142,None
7,291143904be2c9da57d8a924255b6a246ac4ee6429ce7f...,2026-03-05T00:32:38,172.21,True,LLMCacheType,resp_074d76b3cc1d78e50069a8cf11ef1c819bb0539ac...,gpt-5.2-codex,85518,20.903599,None
8,0551ca053b9806b022f89f3f33eaf52a012cc4103db306...,2026-03-05T00:32:17,171.12,True,LLMCacheType,resp_074d76b3cc1d78e50069a8cf0b4d84819bb3807f5...,gpt-5.2-codex,83497,6.382370,None
9,f2497ba9768e272bbc45b482bda3d3128ef08e170c4cc4...,2026-03-05T00:31:59,170.31,True,LLMCacheType,resp_074d76b3cc1d78e50069a8cefaee14819bbee572c...,gpt-5.2-codex,83331,5.034049,None


13b347d1c7235b579b577e164f608993693b24dbeabdfee5b3f52b55f7f1d598.pkl
977ad10dcd77a1b5899e05b816e26c10213abd83f838ad4622b3309df9d51a30.pkl
2675206040f28983647b3612cb91e9cc9fa7e7a9715dd39090d976bb89fb183d.pkl
ebf4fbe7fca036c17b6abc9b56d1299cfc55b98f0d8cdaaf25c06439ff62a27f.pkl
9be1ca4e24afddcdb03035bdfd06ad887d796abfbf80b1f736c2c4bcbabf5aa3.pkl
bfac5f109b8460fe371d90012c17088c35c8727356fa3a406e425f603499192f.pkl
be578ede4f8c75524668755057616f86f11364331417ade5c9db9abecb6e39af.pkl
291143904be2c9da57d8a924255b6a246ac4ee6429ce7fef959cb9cde7974510.pkl
0551ca053b9806b022f89f3f33eaf52a012cc4103db3068d2ef19155e5dd1d26.pkl
f2497ba9768e272bbc45b482bda3d3128ef08e170c4cc40b997401936e4525fc.pkl


In [13]:
# Load the pickle file for the single most recent llm_cache entry
if not cache_files:
    raise RuntimeError("No cache files found in cache_dir")

most_recent_path = cache_files[0]
with open(most_recent_path, "rb") as f:
    most_recent_entry = pickle.load(f)

print(f"Loaded: {most_recent_path}")
print(f"Type: {type(most_recent_entry).__name__}")

# Show key fields if present
print("parent_hash:", getattr(most_recent_entry, "parent_hash", None))
print("llm_time:", getattr(most_recent_entry, "llm_time", None))

resp = most_recent_entry.response

Loaded: /mnt/labstore/bespoke_olap/cache/llm_cache/13b347d1c7235b579b577e164f608993693b24dbeabdfee5b3f52b55f7f1d598.pkl
Type: LLMCacheType
parent_hash: 72d781ba3bc4e77a56823ab965383b954984ae3c
llm_time: 2.980950719676912


In [14]:
for out in resp.output:
    print(out)

ResponseOutputMessage(id='msg_0c76abda3f84b97f0069a9e9a7979c8199b3b5d85bf8977fdd', content=[ResponseOutputText(annotations=[], text='Implemented. I analyzed the workload references and produced a creative in-memory storage layout plan per accessed table, then wrote the final result to:\n\n`storage_plan.txt`', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')


In [15]:
input = [
    {
        "content": "You are an expert database engineer and skilled programmer.\nOptimize a specialized high-performance database engine in C++ that is tailored to only execute a predefined set of queries. The queries to be supported are listed in `queries.txt` (22 queries).\n\nA basic implementation of the database engine is already done in your working directory. \nYour Task is to adapt the query implementation code (`query_impl.cpp`) and pin the query-execution to a single logical CPU core. Do this to ensure deterministic performance measurements and reduce noise from scheduling variability. Use the same core for all implemented queries. \nCPU affinity helpers is predefined in cpu_affinity.hpp.\nYou have to use the following functions, no need to implement them yourself,\nthey are already provided by the runtime:\n\nCPU affinity:\n  Pin the process to a single logical CPU for deterministic execution:\n    void pin_process_to_cpu(int cpu_id);\n\n  Restore affinity to all available CPUs:\n    void unpin_process_from_cpus();\n\nPin to core 3.",
        "role": "user",
    }
]


In [16]:
# Exact cache lookup for the input object defined above
import json

exact_hits = []
for p in cache_files:
    try:
        with open(p, "rb") as f:
            obj = pickle.load(f)
        hp = getattr(obj, "hash_payload", None)
        if not hp:
            continue
        payload = json.loads(hp)
    except Exception:
        continue

    if payload.get("input") == input:
        exact_hits.append((p, obj))

print(f"Exact input matches: {len(exact_hits)}")
if exact_hits:
    print(exact_hits[0])

exact_hits

Exact input matches: 1
(PosixPath('/mnt/labstore/bespoke_olap/cache/llm_cache/c0f945b90aa5f52ef14c90efb2885dc359d26218e6f2d7219510fe79be8ed36a.pkl'), <llm_cache.cached_openai.LLMCacheType object at 0x7bdc7e0c84d0>)


[(PosixPath('/mnt/labstore/bespoke_olap/cache/llm_cache/c0f945b90aa5f52ef14c90efb2885dc359d26218e6f2d7219510fe79be8ed36a.pkl'),
  <llm_cache.cached_openai.LLMCacheType at 0x7bdc7e0c84d0>)]

In [17]:
resp = exact_hits[0][1].response
resp.__dict__

{'id': 'resp_022e94574373c1fb0069a6eb556a388199a7b90fcc333e8bd5',
 'created_at': 1772546901.0,
 'error': None,
 'incomplete_details': None,
 'instructions': 'You can edit files inside output using the apply_patch tool. When modifying an existing file, include the file contents between <BEGIN_FILES> and <END_FILES> in your prompt. You can run shell commands using the shell tool. Do not emit argv form. You can compile the code using the compile tool. You can run a list of queries using the run tool. The run tool automatically compiles the code. You can specify the queries to run and the scale factors to use. If no queries are specified, all queries will be run.',
 'metadata': {},
 'model': 'gpt-5.2-codex',
 'object': 'response',
 'output': [ResponseReasoningItem(id='rs_022e94574373c1fb0069a6eb5613f08199b70c1db09bcef7b9', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
  ResponseFunctionShellToolCall(id='sh_022e94574373c1fb0069a6eb569fdc8199b5c9a2c90a005d

In [18]:
resp.output

[ResponseReasoningItem(id='rs_022e94574373c1fb0069a6eb5613f08199b70c1db09bcef7b9', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionShellToolCall(id='sh_022e94574373c1fb0069a6eb569fdc8199b5c9a2c90a005df5', action=Action(commands=['ls'], max_output_length=None, timeout_ms=None), call_id='call_3GQexGa2LF0cp8lSDro9Koxt', status='completed', type='shell_call', created_by=None, environment=None)]

In [19]:
hash_payload = exact_hits[0][1].hash_payload
print(hash_payload)

{"artifacts_in_context":"","config_kwargs":"max_snapshot_csv_size_mb=5.0,start_snapshot=1bd7dd36ce312f78687c999df06fc43a790329f4","conversation_id":null,"input":[{"content":"You are an expert database engineer and skilled programmer.\nOptimize a specialized high-performance database engine in C++ that is tailored to only execute a predefined set of queries. The queries to be supported are listed in `queries.txt` (22 queries).\n\nA basic implementation of the database engine is already done in your working directory. \nYour Task is to adapt the query implementation code (`query_impl.cpp`) and pin the query-execution to a single logical CPU core. Do this to ensure deterministic performance measurements and reduce noise from scheduling variability. Use the same core for all implemented queries. \nCPU affinity helpers is predefined in cpu_affinity.hpp.\nYou have to use the following functions, no need to implement them yourself,\nthey are already provided by the runtime:\n\nCPU affinity:\n

In [20]:
resp.__dict__

{'id': 'resp_022e94574373c1fb0069a6eb556a388199a7b90fcc333e8bd5',
 'created_at': 1772546901.0,
 'error': None,
 'incomplete_details': None,
 'instructions': 'You can edit files inside output using the apply_patch tool. When modifying an existing file, include the file contents between <BEGIN_FILES> and <END_FILES> in your prompt. You can run shell commands using the shell tool. Do not emit argv form. You can compile the code using the compile tool. You can run a list of queries using the run tool. The run tool automatically compiles the code. You can specify the queries to run and the scale factors to use. If no queries are specified, all queries will be run.',
 'metadata': {},
 'model': 'gpt-5.2-codex',
 'object': 'response',
 'output': [ResponseReasoningItem(id='rs_022e94574373c1fb0069a6eb5613f08199b70c1db09bcef7b9', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
  ResponseFunctionShellToolCall(id='sh_022e94574373c1fb0069a6eb569fdc8199b5c9a2c90a005d